In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("lab02B.ipynb")

In [2]:
import numpy as np
import pandas as pd
%matplotlib inline

### Q1a 使用`groupby`及其聚合方法，创建一个`best_result_percentage_only`的Series，只包含每个`Party`的最佳选举结果，按降序排序。且只包含`%`超过10%的结果

In [3]:
elections = pd.read_csv('data/elections.csv')

In [11]:
best_result_percentage_only = elections[elections['%'] > 10].groupby('Party')['%'].agg('max').sort_values(ascending=False)
best_result_percentage_only

Party
Democratic               61.344703
Republican               60.907806
Democratic-Republican    57.210122
National Union           54.951512
Whig                     53.051213
Liberal Republican       44.071406
National Republican      43.796073
Northern Democratic      29.522311
Progressive              27.457433
American                 21.554001
Independent              18.956298
Southern Democratic      18.138998
American Independent     13.571218
Constitutional Union     12.639283
Free Soil                10.138474
Name: %, dtype: float64

In [12]:
grader.check("q1a")

q1a results: All test cases passed!

### Q1b，重复Q1a，但返回Dataframe而不是Series，即包含所有列<br>
**容易犯错：每列都取max的话会让数据记录间的对应关系出错**<br>
更多解决方案见课程04，1:04:00左右<br>
<img src="planA.png" width = "600"><br>
<img src="planBCD.png" width = "600"><br>
你的理解是对的。如果你用 `groupby('Party').agg('max')`，对于字符串列（如 `Result`），会取每个党派该列的字典序最大值。
所以只要某个党派有过 `Win`，聚合后该党派的所有记录在 `Result` 列都会显示 `Win`；如果从未 `Win` 过，就会显示 `Lose`。

In [25]:
best_result = elections[elections['%'] > 10].sort_values(by='%', ascending=False).groupby('Party').agg('first')
best_result

,Year,Candidate,Popular vote,Result,%
Party,,,,,
American,1856,Millard Fillmore,873053,loss,21.554001
American Independent,1968,George Wallace,9901118,loss,13.571218
Constitutional Union,1860,John Bell,590901,loss,12.639283
Democratic,1964,Lyndon Johnson,43127041,win,61.344703
Democratic-Republican,1824,Andrew Jackson,151271,loss,57.210122
Free Soil,1848,Martin Van Buren,291501,loss,10.138474
Independent,1992,Ross Perot,19743821,loss,18.956298
Liberal Republican,1872,Horace Greeley,2834761,loss,44.071406
National Republican,1828,John Quincy Adams,500897,loss,43.796073


In [26]:
grader.check("q1b")

q1b results: All test cases passed!

In [27]:
# Run this cell to print the last four rows; no further action is needed.
elections.tail(4)

,Year,Candidate,Party,Popular vote,Result,%
178,2020,Joseph Biden,Democratic,81268924,win,51.311515
179,2020,Donald Trump,Republican,74216154,loss,46.858542
180,2020,Jo Jorgensen,Libertarian,1865724,loss,1.177979
181,2020,Howard Hawkins,Green,405035,loss,0.255731


只关注至少获胜过一次的Party，使用`groupby.filter()`函数

In [28]:
# Run this cell to keep only the rows of parties that have
# elected a president; no further action is needed.
def at_least_one_candidate_in_the_frame_has_won(frame):
    """Returns df with rows only kept for parties that have
    won at least one election
    """
    return (frame["Result"] == 'win').any()

winners_only = (
    elections
        .groupby("Party")
        .filter(at_least_one_candidate_in_the_frame_has_won)
)
winners_only.tail(5)

,Year,Candidate,Party,Popular vote,Result,%
171,2012,Mitt Romney,Republican,60933504,loss,47.384076
173,2016,Donald Trump,Republican,62984828,win,46.407862
176,2016,Hillary Clinton,Democratic,65853514,loss,48.521539
178,2020,Joseph Biden,Democratic,81268924,win,51.311515
179,2020,Donald Trump,Republican,74216154,loss,46.858542


使用`lambda`匿名函数而不是`def`显式定义函数

In [30]:
winners_only = (
    elections
    .groupby("Party")
    .filter(lambda x : (x["Result"] == 'win').any())
)
winners_only.tail(5)

,Year,Candidate,Party,Popular vote,Result,%
171,2012,Mitt Romney,Republican,60933504,loss,47.384076
173,2016,Donald Trump,Republican,62984828,win,46.407862
176,2016,Hillary Clinton,Democratic,65853514,loss,48.521539
178,2020,Joseph Biden,Democratic,81268924,win,51.311515
179,2020,Donald Trump,Republican,74216154,loss,46.858542


### Q1c 使用`filter`创建DataFrame`major_party_result_since_1988`，要求：包含1988年后所有 所属 Party在1988年以来的选举中曾获得1%以上 的候选记录

In [37]:
# results_since_1988 = elections[elections['Year'] > 1988]
major_party_results_since_1988 = elections[elections['Year'] > 1988].groupby('Party').filter(lambda x : (x['%'].max() >= 1))
major_party_results_since_1988.head()

,Year,Candidate,Party,Popular vote,Result,%
139,1992,Andre Marrou,Libertarian,290087,loss,0.278516
140,1992,Bill Clinton,Democratic,44909806,win,43.118485
142,1992,George H. W. Bush,Republican,39104550,loss,37.544784
143,1992,Ross Perot,Independent,19743821,loss,18.956298
144,1996,Bill Clinton,Democratic,47400125,win,49.296938


In [38]:
grader.check("q1c")

q1c results: All test cases passed!

### Q2 使用`.str.split`创建名为`elections_with_first_name`的新dataframe，其中新列'Fist Name'是候选人的名

In [42]:
elections_with_first_name = elections.copy()
elections_with_first_name['First Name'] = elections_with_first_name['Candidate'].str.split().str[0]
elections_with_first_name

,Year,Candidate,Party,Popular vote,Result,%,First Name
0,1824,Andrew Jackson,Democratic-Republican,151271,loss,57.210122,Andrew
1,1824,John Quincy Adams,Democratic-Republican,113142,win,42.789878,John
2,1828,Andrew Jackson,Democratic,642806,win,56.203927,Andrew
3,1828,John Quincy Adams,National Republican,500897,loss,43.796073,John
4,1832,Andrew Jackson,Democratic,702735,win,54.574789,Andrew
...,...,...,...,...,...,...,...
177,2016,Jill Stein,Green,1457226,loss,1.073699,Jill
178,2020,Joseph Biden,Democratic,81268924,win,51.311515,Joseph
179,2020,Donald Trump,Republican,74216154,loss,46.858542,Donald
180,2020,Jo Jorgensen,Libertarian,1865724,loss,1.177979,Jo


In [43]:
grader.check("q2")

q2 results: All test cases passed!

加载来自加州的`babynames`

In [44]:
file_path = 'data/namesbystate_ca.txt.gz'
column_labels = ['State', 'Sex', 'Year', 'Name', 'Count']

babynames = pd.read_csv(file_path, names=column_labels)

babynames.head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


包含2022年以来所有名字出现频率的表。

In [45]:
# Run this cell to create a table with the frequency
# of all names from 2022; no further action is needed.
babynames_2022 = (
    babynames[babynames['Year'] == 2022]
              .groupby("Name")
              .sum()[["Count"]]
              .reset_index()
)
babynames_2022

,Name,Count
0,Aadhini,6
1,Aadhira,5
2,Aadhya,33
3,Aadi,11
4,Aadit,5
...,...,...
6196,Zyla,20
6197,Zylah,14
6198,Zylo,5
6199,Zyon,17


### Q3 使用`merge`合并`babynames_2022`和`elections_with_first_name`<br>

In [46]:
presidential_candidates_and_name_popularity = pd.merge(left=babynames_2022, right=elections_with_first_name, left_on='Name', right_on='First Name')
presidential_candidates_and_name_popularity

,Name,Count,Year,Candidate,Party,Popular vote,Result,%,First Name
0,Aaron,903,1920,Aaron S. Watkins,Prohibition,188787,loss,0.708351,Aaron
1,Abraham,353,1860,Abraham Lincoln,Republican,1855993,win,39.699408,Abraham
2,Abraham,353,1864,Abraham Lincoln,National Union,2211317,win,54.951512,Abraham
3,Allan,38,1916,Allan L. Benson,Socialist,590524,loss,3.194193,Allan
4,Andre,242,1992,Andre Marrou,Libertarian,290087,loss,0.278516,Andre
...,...,...,...,...,...,...,...,...,...
139,William,831,1932,William Z. Foster,Communist,103307,loss,0.261069,William
140,William,831,1936,William Lemke,Union,892378,loss,1.960733,William
141,Woodrow,6,1912,Woodrow Wilson,Democratic,6296284,win,41.933422,Woodrow
142,Woodrow,6,1916,Woodrow Wilson,Democratic,9126868,win,49.367987,Woodrow


In [47]:
grader.check("q3")

q3 results: All test cases passed!

### 使用`presidential_candidates_and_name_popularity`创建一个表`party_result_popular_vote_pivot`，其index是`Party`，其columns是`Result`。每个单元格应包含收到的普选票数总数

In [49]:
party_result_popular_vote_pivot = presidential_candidates_and_name_popularity.pivot_table(index='Party', columns='Result', values='Popular vote', aggfunc='sum', fill_value=0)
party_result_popular_vote_pivot

Result,loss,win
Party,,
American,158271,0
American Independent,11172260,0
Anti-Masonic,100715,0
Anti-Monopoly,134294,0
Citizens,233052,0
Communist,103307,0
Constitution,346721,0
Constitutional Union,590901,0
Democratic,314190254,441751531


In [50]:
grader.check("q4")

q4 results: All test cases passed!